**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [4]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
processed_data_dir = "/dlabdata1/boulenge/wikidumps_data/full_pages_experiement"
processes = 48
max_samples_per_lang = 100
langs = ["simple"]  # Example languages
side_context_sizes = [5]  # Example context sizes
second_month = "20231001"
third_month = "20231101"

**Defining data paths**

In [7]:
# Define directories
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(processed_data_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)

**Prepare files to be parsed**

In [9]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        
    subprocess.run([
        "python", "generate_revisions_file.py",
        "--versions_file", f"{second_month_path}/eval/link_versions",
        "--html_dir", f"{second_month_path}/eval/pages",
        "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
        "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
        "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
        "--max_samples", str(max_samples_per_lang),
        "--save_path", save_path
    ])

    revisions = pd.read_parquet(save_path)
    display(revisions.head(5))

Processing language: simple
Fixing target titles
Creating revisions dictionary
Expected links: 5175


Checking revisions: 100%|██████████| 1841/1841 [00:04<00:00, 450.41it/s]


,first_version,source_title,source_ID,versions
0,9114883,List_of_mayors_in_Norway_2023%E2%80%932027,1052251,[9114892]
1,9126470,Patricia_Greene,818327,[9126510]
2,8833916,Plow,38298,[9150709]
3,9085887,M%C3%A1rio_Zagallo,178441,[9106354]
4,9138325,Deaths_in_2023,986335,[9138326]


**Crawling first versions only**

In [20]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--crawl_second_versions", "False",
        "future_colored_html_path", os.path.join(processed_data_dir, "colored_pages", lang)
    ])

Processing language: simple
Total number of pages to crawl: 200
Running crawling on 48 processes
Parallel multiprocess


**Parsing of the htmls**

In [17]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--non_render_html_dir", f"{second_month_path}/eval/pages",
        "--output_dir", output_dir,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(output_dir)
    sentences_path = list(filter(lambda x: x.endswith("sentences.jsonl"), parsed_htmls_paths))[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5))   

Processing language: simple
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 6.725069046020508
Writing sentences and links v2
Time taken for v2: 6.283459901809692


,md5,id,text,startIdx,endIdx,section,locList
0,b83fb7dd3c7626edd8e85d9ebd8bfb9b,0,"Onika Tanya Maraj-Petty (born December 8, 1982...",71112,71997,Lead,"[{'start': 71111, 'end': 71112}, {'start': 711..."
1,48ee843085e28c3615de0c8022925c96,1,Minaj became the first and only female rapper ...,71998,72347,Lead,"[{'start': 71996, 'end': 71997}, {'start': 720..."
2,a9f862953b3a143d38bc9ae300740ef0,2,"In August 2009, Minaj signed to Young Money En...",72637,72739,Career,"[{'start': 72636, 'end': 72637}, {'start': 726..."
3,2ce122f1150034098fff0517e173ea22,3,"Her debut album, Pink Friday, was released on ...",72740,72862,Career,"[{'start': 72738, 'end': 72756}, {'start': 728..."
4,6acd1e461c528aabddb3760712181d8d,4,It climbed to number one on the US Billboard 2...,72863,73255,Career,"[{'start': 72861, 'end': 72897}, {'start': 729..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/Norwegians,66685,Norwegian,66764,Lead,0
1,file:///wiki/Politician,66765,politician,66825,Lead,0
2,file:///wiki/Conservative_Party_of_Norway,66854,Conservative Party,66978,Lead,1
3,file:///wiki/Prime_Minister_of_Norway,67008,Prime Minister of Norway,67110,Lead,2
4,file:///wiki/2021_Norwegian_parliamentary_elec...,67248,September 2021,67366,Lead,2


**Preprocessing**

In [22]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "modular_preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--n_processes", str(processes)
        ])

        preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        preprocessed_data = pd.read_parquet(preprocessed_data_path)
        display(preprocessed_data.head(5))

Processing language: simple with side context size 5
Running on 48 processes
Parallel multiprocess


100%|██████████| 10/10 [00:04<00:00,  2.37it/s]


The length of the output is 31


100%|██████████| 31/31 [00:00<00:00, 9672.92it/s]


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts
0,List_of_mayors_in_Norway_2023%E2%80%932027,1052251,M%C3%B8re_og_Romsdal,None,[26],None,"Hustadvika. KrF, and Centre Party; Nils Christ...",Møre og Romsdal,Lead,None,"In September and October 2023, every municipal...",9114883,9114892,None,missing_span,"[{'context': 'In September and October 2023, e..."
1,Puffin_Rock,954730,Disney_XD,None,[1],None,It follows the adventures of two young puffins...,Disney XD,Lead,None,Puffin Rock is a Irish animated children's tel...,8673424,9134125,None,missing_mention_with_sentence_modifications,[{'context': 'Puffin Rock is a Irish animated ...
2,Kevin_Costner,73022,"Lynwood,_California",None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,"Lynwood, California",Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...
3,Kevin_Costner,73022,United_States,None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,USA,Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...
4,Kevin_Costner,73022,Eliot_Ness,None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,Eliot Ness,Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...


**Merging**

In [42]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "modular_merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
print(scored_files_example)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps
Processing language simple (1/1)
Finished processing language simple
Processing language simple (1/1)
Loaded pages
Loaded redirects


100%|██████████| 238680/238680 [00:00<00:00, 646135.90it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 31 links for language simple
Removed 0 links because of empty contexts
Removed 2 links because target title is not in source titles
Removed 5 links because of missing section
After cleaning, there are 24 links for language simple
Finished processing language simple
['mention_map_simple.parquet', 'simple_5.parquet']


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,List_of_mayors_in_Norway_2023%E2%80%932027,1052251,M%C3%B8re_og_Romsdal,None,[26],None,"Hustadvika. KrF, and Centre Party; Nils Christ...",Møre og Romsdal,Lead,None,"In September and October 2023, every municipal...",9114883,9114892,None,missing_span,"[{'context': 'In September and October 2023, e...",Møre og Romsdal is a county in Norway. Norway ...
1,Puffin_Rock,954730,Disney_XD,None,[1],None,It follows the adventures of two young puffins...,Disney XD,Lead,None,Puffin Rock is a Irish animated children's tel...,8673424,9134125,None,missing_mention_with_sentence_modifications,[{'context': 'Puffin Rock is a Irish animated ...,Disney XD is an American digital cable and sat...
2,Kevin_Costner,73022,United_States,None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,USA,Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...,"The United States of America, commonly known a..."
3,Kevin_Costner,73022,Eliot_Ness,None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,Eliot Ness,Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...,Eliot Ness was an American Prohibition agent. ...
4,Kevin_Costner,73022,Academy_Award_for_Best_Director,None,[0],None,{{Infobox actor. | name = Kevin Costner. | ima...,Academy Award for Best Director,Lead,None,"Kevin Michael Costner (born January 18, 1955) ...",9078665,9093331,None,missing_span,[{'context': 'Kevin Michael Costner (born Janu...,"The Academy Award for Best Directing, usually ..."


**Scoring the contexts**

In [39]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

subprocess.run([
    "python", "get_scores.py",
    "--data_dir", data_dir,
    "--output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--models_dir", "/dlabdata1/tsoares/models",
    "--models_prefix", "roberta_full",
    "--use_section_title",
    "--use_mentions",
    "--use_cuda",
    "--only_multilingual",
    "--multilingual_name", "multilingual-all"
])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

context:
In September and October 2023, every municipal council in Norway, is supposed to elect a mayor. Four years later, the municipal councils will. elect mayors, again. (Names were added to this page, as late as October 7.) In regard to Alstahaug municipality, media says that there is a written[1] agreement, that makes Peter Talseth the new mayor [when voting will be done at the next council meeting].[2] Furthermore, the municipality is in Nordland (county). No mayor has been elected (2023) in these municipalities: Alta, (municipality), Alvdal, Alver, Andøy, Aremark, Arendal, Asker, Askvoll, Askøy, Aukra, Aure, Aurland, Aurskog-Høland, Austevoll, Austrheim, Averøy, Balsfjord

middle_sentence:
{'clean_sentence': 'In September and October 2023, every municipal council in Norway,', 'clean_sentence2': None, 'endIdx1': 34966, 'endIdx2': 34966.0, 'ids1': array([0]), 'ids2': array([0]), 'index': 0, 'locList1': array([{'end': 34884, 'start': 34847}, {'end': 34951, 'start': 34934},
       {

**Colouring the pages**

In [54]:
output_dir = os.path.join(processed_data_dir, "colored_pages")

# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", f"{second_month_path}/eval/pages",
    "--output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
colored_files_example = os.listdir(output_dir)
colored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), colored_files_example))
colored_data_path = os.path.join(output_dir, colored_files_example[0], "missing_mention")

example_data_path = os.listdir(colored_data_path)[0]  
print("\nExample (click to open):")
print(os.path.join(output_dir, colored_files_example[0], "missing_mention", example_data_path))



Processing language simple with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 24/24 [00:02<00:00, 11.69it/s]



Example (click to open):
/dlabdata1/boulenge/wikidumps_data/colored_pages/simple_5/missing_mention/1041893_9030083_scottish.html


**Locally download the colored pages**

In [62]:
user_cluster  = "boulenge@iccluster039"
local_path = "Desktop/htmls"

scp_command = f"scp -qr {user_cluster}:{output_dir} {local_path} > /dev/null"

#subprocess.run(scp_command, shell=True)

print(f"Run the following in the terminal to dowload the colored pages:\n{scp_command}")

Run the following in the terminal to dowload the colored pages:
scp -qr boulenge@iccluster039:/dlabdata1/boulenge/wikidumps_data/colored_pages Desktop/htmls > /dev/null
